In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

## Instalação de Dependências

In [ ]:

!pip install -q transformers peft torch sacrebleu rouge-score nltk pandas \
             matplotlib seaborn tqdm

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('Todas as dependências instaladas com sucesso!')

In [ ]:
%pip install \
    "fastapi>=0.110.0" \
    "uvicorn[standard]>=0.29.0" \
    "transformers>=4.40.0" \
    "torch>=2.2.0" \
    "accelerate>=0.29.0" \
    "pydantic>=2.0.0" \
    "python-multipart>=0.0.9" \
    "sentencepiece>=0.2.0"

## Importações

In [ ]:
import json
import math
import re
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm import tqdm

# Transformers & PEFT
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

# Métricas NLP
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from nltk.tokenize import word_tokenize

warnings.filterwarnings('ignore')

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('Importações concluídas!')
print(f'   PyTorch: {torch.__version__}')
print(f'   Dispositivo disponível: {"GPU (CUDA)" if torch.cuda.is_available() else "CPU"}')

In [ ]:
!pip install --upgrade torchao

## Carregando dataset

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/dataset.jsonl'

samples = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

print(f'✅ Dataset carregado com {len(samples)} amostras.\n')

# Exibição amigável das primeiras amostras
df_raw = pd.DataFrame(samples)
df_raw[['Instruction', 'Output']].head(3).style \
    .set_caption('Primeiras amostras do dataset') \
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})

## Carregamento dos modelos fine-tunados (Rodar um por vez)

### IBM Causal

In [ ]:
MODEL_BASE      = 'ibm-granite/granite-3.3-2b-instruct'
LORA_MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/TreinoIBM/lora_ibm_causal_finetuned_model'
TOKENIZER_PATH  = '/content/drive/MyDrive/Colab Notebooks/TreinoIBM/ibm_tokenizer'

print('⏳ Carregando tokenizer...')
finetuned_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# GPT-2 não tem pad_token por padrão — usamos o eos_token
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
    print('   ⚠️  pad_token não encontrado — usando eos_token como pad_token.')

print('⏳ Carregando modelo base...')
base_model = AutoModelForCausalLM.from_pretrained(MODEL_BASE)

print('⏳ Aplicando adaptadores LoRA...')
finetuned_model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)
finetuned_model.eval()  # Modo de inferência (desativa dropout)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
finetuned_model = finetuned_model.to(DEVICE)

# Contagem de parâmetros
total_params    = sum(p.numel() for p in finetuned_model.parameters())
trainable_params = sum(p.numel() for p in finetuned_model.parameters() if p.requires_grad)

print(f'\n✅ Modelo carregado no dispositivo: {DEVICE}')
print(f'   Parâmetros totais   : {total_params:,}')
print(f'   Parâmetros LoRA     : {trainable_params:,}')
print(f'   Redução LoRA        : {(1 - trainable_params/total_params)*100:.1f}% do total')

### Tucano Causal

In [ ]:
MODEL_BASE      = 'TucanoBR/Tucano-1b1'
LORA_MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/Treino tucano/lora_tucano_causal_finetuned_model'
TOKENIZER_PATH  = '/content/drive/MyDrive/Colab Notebooks/Treino tucano/tucano_tokenizer'

print('⏳ Carregando tokenizer...')
finetuned_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# GPT-2 não tem pad_token por padrão — usamos o eos_token
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
    print('   ⚠️  pad_token não encontrado — usando eos_token como pad_token.')

print('⏳ Carregando modelo base...')
base_model = AutoModelForCausalLM.from_pretrained(MODEL_BASE)

print('⏳ Aplicando adaptadores LoRA...')
finetuned_model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)
finetuned_model.eval()  # Modo de inferência (desativa dropout)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
finetuned_model = finetuned_model.to(DEVICE)

# Contagem de parâmetros
total_params    = sum(p.numel() for p in finetuned_model.parameters())
trainable_params = sum(p.numel() for p in finetuned_model.parameters() if p.requires_grad)

print(f'\n✅ Modelo carregado no dispositivo: {DEVICE}')
print(f'   Parâmetros totais   : {total_params:,}')
print(f'   Parâmetros LoRA     : {trainable_params:,}')
print(f'   Redução LoRA        : {(1 - trainable_params/total_params)*100:.1f}% do total')

### Google Seq2seq

In [ ]:
MODEL_BASE      = 'google/mt5-base'
LORA_MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/Treino_google/lora_google_sequencial_finetuned_model'
TOKENIZER_PATH  = '/content/drive/MyDrive/Colab Notebooks/Treino_google/google_tokenizer'

print('⏳ Carregando tokenizer...')
finetuned_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# GPT-2 não tem pad_token por padrão — usamos o eos_token
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
    print('   ⚠️  pad_token não encontrado — usando eos_token como pad_token.')

print('⏳ Carregando modelo base...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_BASE)

print('⏳ Aplicando adaptadores LoRA...')
finetuned_model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)
finetuned_model.eval()  # Modo de inferência (desativa dropout)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
finetuned_model = finetuned_model.to(DEVICE)

# Contagem de parâmetros
total_params    = sum(p.numel() for p in finetuned_model.parameters())
trainable_params = sum(p.numel() for p in finetuned_model.parameters() if p.requires_grad)

print(f'\n✅ Modelo carregado no dispositivo: {DEVICE}')
print(f'   Parâmetros totais   : {total_params:,}')
print(f'   Parâmetros LoRA     : {trainable_params:,}')
print(f'   Redução LoRA        : {(1 - trainable_params/total_params)*100:.1f}% do total')

### Unicamp Seq2seq

In [ ]:
MODEL_BASE      = 'unicamp-dl/ptt5-v2-large'
LORA_MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/Treino_unicamp/lora_unicamp_sequencial_finetuned_model'
TOKENIZER_PATH  = '/content/drive/MyDrive/Colab Notebooks/Treino_unicamp/unicamp_tokenizer'

print('⏳ Carregando tokenizer...')
finetuned_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# GPT-2 não tem pad_token por padrão — usamos o eos_token
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
    print('   ⚠️  pad_token não encontrado — usando eos_token como pad_token.')

print('⏳ Carregando modelo base...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_BASE)

print('⏳ Aplicando adaptadores LoRA...')
finetuned_model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)
finetuned_model.eval()  # Modo de inferência (desativa dropout)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
finetuned_model = finetuned_model.to(DEVICE)

# Contagem de parâmetros
total_params    = sum(p.numel() for p in finetuned_model.parameters())
trainable_params = sum(p.numel() for p in finetuned_model.parameters() if p.requires_grad)

print(f'\n✅ Modelo carregado no dispositivo: {DEVICE}')
print(f'   Parâmetros totais   : {total_params:,}')
print(f'   Parâmetros LoRA     : {trainable_params:,}')
print(f'   Redução LoRA        : {(1 - trainable_params/total_params)*100:.1f}% do total')

## Função de Geração de Respostas

In [ ]:
# ============================================================
#  FUNÇÃO DE GERAÇÃO DE TEXTO
# ============================================================

def build_prompt(instruction: str, input_text: str = '') -> str:
    """
    Constrói o prompt no formato instruction-following.
    Se houver contexto adicional, ele é incluído após ### Input.
    """
    if input_text.strip():
        return (
            f'### Instruction:\n{instruction}\n\n'
            f'### Input:\n{input_text}\n\n'
            f'### Response:\n'
        )
    return (
        f'### Instruction:\n{instruction}\n\n'
        f'### Response:\n'
    )


def generate_response(
    model,
    tokenizer,
    instruction: str,
    input_text: str = '',
    max_new_tokens: int = 200,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    """
    Gera a resposta do modelo dado instruction + input.

    Parâmetros
    ----------
    max_new_tokens : int
        Número máximo de tokens a gerar além do prompt.
    temperature : float
        Controla a aleatoriedade (0 = determinístico, 1 = máximo).
    top_p : float
        Nucleus sampling: considera apenas tokens cuja probabilidade
        cumulativa atinge top_p.

    Retorna
    -------
    str
        Apenas a parte gerada (sem o prompt de entrada).
    """
    prompt = build_prompt(instruction, input_text)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove os tokens do prompt — retorna apenas a geração
    prompt_len = inputs['input_ids'].shape[1]
    generated_ids = output_ids[0][prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


print('✅ Funções de geração definidas!')

# --- TESTE RÁPIDO ---
sample = samples[0]
test_response = generate_response(
    finetuned_model, finetuned_tokenizer,
    sample['Instruction'], ''
)
print(f'\nTeste rápido com a primeira amostra:')
print(f'    Instrução : {sample["Instruction"]}')
print(f'    Contexto  : {sample["Output"]}')
print(f'    Esperado  : {sample["Output"][:80]}...')
print(f'    Gerado    : {test_response[:80]}...')

## Métricas

### Perplexidade (PPL)

In [ ]:
# ============================================================
#  MÉTRICA 1 — PERPLEXIDADE (PPL)
# ============================================================

def compute_perplexity_for_sample(
    model, tokenizer, instruction: str, input_text: str, reference: str
) -> float:
    """
    Calcula a Perplexidade do modelo sobre o texto de referência
    condicionado ao prompt (instruction + input).

    A perplexidade é derivada da loss de cross-entropy:
        loss = -mean(log P(token_i | contexto))
        PPL  = exp(loss)
    """
    prompt   = build_prompt(instruction, input_text)
    full_text = prompt + reference

    encodings = tokenizer(
        full_text,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    # Comprimento do prompt em tokens (não queremos calcular loss sobre ele)
    prompt_tokens = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=512
    )['input_ids'].shape[1]

    input_ids = encodings['input_ids']

    # Labels: -100 para tokens do prompt (ignorados na loss)
    labels = input_ids.clone()
    labels[0, :prompt_tokens] = -100

    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss  # cross-entropy média sobre os tokens de resposta

    return math.exp(loss.item())


# Calcula PPL para todas as amostras
print('⏳ Calculando Perplexidade para todas as amostras...')
ppl_scores = []
for s in tqdm(samples, desc='PPL'):
    ppl = compute_perplexity_for_sample(
        finetuned_model, finetuned_tokenizer,
        s['Instruction'],'',s['Output']
    )
    ppl_scores.append(ppl)

mean_ppl = np.mean(ppl_scores)
print(f'\n📊 Perplexidade Média : {mean_ppl:.2f}')
print(f'   Mínima            : {min(ppl_scores):.2f}  (amostra mais fácil)')
print(f'   Máxima            : {max(ppl_scores):.2f}  (amostra mais difícil)')
print(f'\n💡 Interpretação: PPL = {mean_ppl:.1f} significa que, em média,')
print(f'   o modelo considera ~{mean_ppl:.0f} próximas palavras igualmente prováveis.')

### Bilingual Evaluation Understudy (BLEU)

In [ ]:
# ============================================================
#  MÉTRICA 2 — BLEU
# ============================================================

def compute_bleu(hypotheses: list, references: list) -> dict:
    """
    Calcula BLEU usando sacrebleu.

    Parâmetros
    ----------
    hypotheses : list[str]   — textos gerados pelo modelo
    references : list[str]   — textos de referência (ground truth)

    Retorna
    -------
    dict com score geral e scores por n-grama
    """
    bleu = BLEU(effective_order=True)
    # sacrebleu espera: hyps = list[str], refs = list[list[str]]
    result = bleu.corpus_score(hypotheses, [references])

    return {
        'bleu_score' : result.score,
        '1-gram'     : result.precisions[0],
        '2-gram'     : result.precisions[1],
        '3-gram'     : result.precisions[2],
        '4-gram'     : result.precisions[3],
        'brevity_penalty': result.bp,
    }


# Gera todas as respostas
print('⏳ Gerando respostas do modelo para calcular BLEU...')
generated_responses = []
reference_outputs   = []

for s in tqdm(samples, desc='Gerando'):
    gen = generate_response(
        finetuned_model, finetuned_tokenizer,
        s['Instruction'], ''
    )
    generated_responses.append(gen)
    reference_outputs.append(s['Output'])

bleu_results = compute_bleu(generated_responses, reference_outputs)

print(f'\n📊 Resultados BLEU:')
print(f'   BLEU Score (corpus)  : {bleu_results["bleu_score"]:.2f}')
print(f'   Precisão 1-gram      : {bleu_results["1-gram"]:.2f}%')
print(f'   Precisão 2-gram      : {bleu_results["2-gram"]:.2f}%')
print(f'   Precisão 3-gram      : {bleu_results["3-gram"]:.2f}%')
print(f'   Precisão 4-gram      : {bleu_results["4-gram"]:.2f}%')
print(f'   Brevity Penalty      : {bleu_results["brevity_penalty"]:.4f}')
print(f'\n💡 BLEU > 30 é geralmente considerado bom para geração de texto.')

### Recall-Oriented Understudy for Gisting Evaluation (ROUGE)

In [ ]:
# ============================================================
#  MÉTRICA 3 — ROUGE
# ============================================================

def compute_rouge_scores(hypotheses: list, references: list) -> pd.DataFrame:
    """
    Calcula ROUGE-1, ROUGE-2 e ROUGE-L para cada par (gerado, referência).

    Retorna um DataFrame com Precision, Recall e F1 para cada variante.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    records = []

    for hyp, ref in zip(hypotheses, references):
        scores = scorer.score(ref, hyp)
        records.append({
            'ROUGE-1 P' : scores['rouge1'].precision,
            'ROUGE-1 R' : scores['rouge1'].recall,
            'ROUGE-1 F1': scores['rouge1'].fmeasure,
            'ROUGE-2 P' : scores['rouge2'].precision,
            'ROUGE-2 R' : scores['rouge2'].recall,
            'ROUGE-2 F1': scores['rouge2'].fmeasure,
            'ROUGE-L P' : scores['rougeL'].precision,
            'ROUGE-L R' : scores['rougeL'].recall,
            'ROUGE-L F1': scores['rougeL'].fmeasure,
        })

    return pd.DataFrame(records)


df_rouge = compute_rouge_scores(generated_responses, reference_outputs)

print('📊 Médias ROUGE (todas as amostras):\n')
rouge_means = df_rouge.mean()

for variant in ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']:
    p  = rouge_means[f'{variant} P']
    r  = rouge_means[f'{variant} R']
    f1 = rouge_means[f'{variant} F1']
    print(f'   {variant:<8}  Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}')

print(f'\n💡 ROUGE-L F1 alto indica que o modelo mantém a ordem e estrutura do texto de referência.')

### Fidelidade (Faithfulness)

In [ ]:
# ============================================================
#  MÉTRICA 4 — FIDELIDADE (FAITHFULNESS)
# ============================================================

def extract_key_tokens(text: str) -> set:
    """
    Extrai tokens relevantes: números, palavras com maiúscula inicial
    (marcas, modelos), e tokens longos (substantivos específicos).
    Ignora stopwords e tokens muito curtos.
    """
    STOPWORDS = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
        'could', 'should', 'may', 'might', 'to', 'of', 'in', 'on',
        'at', 'for', 'with', 'by', 'from', 'or', 'and', 'but', 'if',
        'your', 'my', 'our', 'their', 'this', 'that', 'it', 'its',
    }
    tokens = re.findall(r'[A-Za-z0-9]+', text.lower())
    # Mantém números, tokens com >= 4 chars fora de stopwords
    return {
        t for t in tokens
        if (t.isdigit() or len(t) >= 4) and t not in STOPWORDS
    }


def compute_faithfulness(instruction: str, input_text: str, generated: str) -> float:
    """
    Calcula a Fidelidade como a proporção de tokens-chave do contexto
    (instruction + input) que aparece na resposta gerada.

    Score = |tokens_contexto ∩ tokens_gerado| / |tokens_contexto|

    Retorna 0.5 (neutro) se não houver tokens-chave no contexto.
    """
    context_tokens   = extract_key_tokens(instruction + ' ' + input_text)
    generated_tokens = extract_key_tokens(generated)

    if not context_tokens:
        return 0.5  # sem contexto mensurável, score neutro

    overlap = context_tokens & generated_tokens
    return len(overlap) / len(context_tokens)


faithfulness_scores = []
for s, gen in zip(samples, generated_responses):
    score = compute_faithfulness(s['Instruction'], '', gen)
    faithfulness_scores.append(score)

mean_faith = np.mean(faithfulness_scores)
print(f'📊 Fidelidade (Faithfulness):')
print(f'   Média  : {mean_faith:.3f}')
print(f'   Mínima : {min(faithfulness_scores):.3f}')
print(f'   Máxima : {max(faithfulness_scores):.3f}')
print(f'\n💡 Score próximo de 1.0 indica que a resposta cita informações do contexto fornecido.')
print(f'   Crítico em sistemas RAG onde o modelo deve se basear nos documentos recuperados.')

### Relevância da Resposta (Answer Relevance)

In [ ]:
# ============================================================
#  MÉTRICA 5 — RELEVÂNCIA DA RESPOSTA (ANSWER RELEVANCE)
# ============================================================

def compute_answer_relevance(instruction: str, generated: str) -> float:
    """
    Calcula a Relevância como Jaccard Similarity entre os tokens-chave
    da instrução e da resposta gerada.

    Jaccard = |A ∩ B| / |A ∪ B|

    Um score alto indica que a resposta usa vocabulário ligado à pergunta.
    """
    instr_tokens = extract_key_tokens(instruction)
    gen_tokens   = extract_key_tokens(generated)

    if not instr_tokens and not gen_tokens:
        return 0.5

    intersection = instr_tokens & gen_tokens
    union        = instr_tokens | gen_tokens

    # Jaccard puro penaliza respostas muito longas — adicionamos bônus
    # de cobertura da instrução para valorizar completude
    jaccard  = len(intersection) / len(union)
    coverage = len(intersection) / len(instr_tokens) if instr_tokens else 0

    return (jaccard + coverage) / 2.0


relevance_scores = [
    compute_answer_relevance(s['Instruction'], gen)
    for s, gen in zip(samples, generated_responses)
]

mean_rel = np.mean(relevance_scores)
print(f'📊 Relevância da Resposta:')
print(f'   Média  : {mean_rel:.3f}')
print(f'   Mínima : {min(relevance_scores):.3f}')
print(f'   Máxima : {max(relevance_scores):.3f}')
print(f'\n💡 Score alto = resposta usa o vocabulário da pergunta e cobre seus aspectos principais.')

### Aderência ao Plano

In [ ]:
# ============================================================
#  MÉTRICA 6 — ADERÊNCIA AO PLANO (PLAN ADHERENCE)
# ============================================================

def detect_structural_elements(text: str) -> dict:
    """
    Detecta elementos estruturais no texto:
    - has_numbered_list : tem lista numerada (1. 2. ...)
    - has_bullets       : tem bullet points (- ou •)
    - has_technical     : tem valores técnicos (números + unidade)
    - has_sections      : tem seções com dois-pontos no início de linha
    - step_count        : número de passos numerados
    """
    return {
        'has_numbered_list': bool(re.search(r'^\d+\.\s', text, re.MULTILINE)),
        'has_bullets'      : bool(re.search(r'^[\-•]\s', text, re.MULTILINE)),
        'has_technical'    : bool(re.search(
            r'\d+[,.]?\d*\s*(psi|mph|lbs|miles|km|liter|L|V\d|octane|months?)',
            text, re.IGNORECASE
        )),
        'has_sections'     : bool(re.search(r'^[A-Z][^\n]+:\s*$', text, re.MULTILINE)),
        'step_count'       : len(re.findall(r'^\d+\.\s', text, re.MULTILINE)),
    }


def compute_plan_adherence(reference: str, generated: str) -> float:
    """
    Calcula a Aderência ao Plano comparando os elementos estruturais
    do texto de referência com os da resposta gerada.

    Para cada elemento estrutural presente na referência, verifica
    se também está presente na resposta gerada.

    Retorna a proporção de elementos seguidos corretamente.
    """
    ref_struct = detect_structural_elements(reference)
    gen_struct = detect_structural_elements(generated)

    checks = []
    binary_features = ['has_numbered_list', 'has_bullets', 'has_technical', 'has_sections']

    for feat in binary_features:
        if ref_struct[feat]:  # Só avalia se a referência tem esse elemento
            checks.append(1.0 if gen_struct[feat] else 0.0)

    # Verifica se contagem de passos é similar (±1)
    if ref_struct['step_count'] > 0:
        step_ratio = min(gen_struct['step_count'], ref_struct['step_count']) / \
                     max(gen_struct['step_count'], ref_struct['step_count'], 1)
        checks.append(step_ratio)

    return np.mean(checks) if checks else 0.5


plan_scores = [
    compute_plan_adherence(s['Output'], gen)
    for s, gen in zip(samples, generated_responses)
]

mean_plan = np.mean(plan_scores)
print(f'📊 Aderência ao Plano (Plan Adherence):')
print(f'   Média  : {mean_plan:.3f}')
print(f'   Mínima : {min(plan_scores):.3f}')
print(f'   Máxima : {max(plan_scores):.3f}')
print(f'\n💡 Score alto indica que o modelo segue a mesma estrutura de resposta da referência:')
print(f'   listas numeradas, bullets, valores técnicos e organização em seções.')

## Tabela Consolidada de Resultados

In [ ]:
# ============================================================
#  TABELA CONSOLIDADA DE RESULTADOS
# ============================================================

rouge_f1_1 = df_rouge['ROUGE-1 F1'].tolist()
rouge_f1_2 = df_rouge['ROUGE-2 F1'].tolist()
rouge_f1_L = df_rouge['ROUGE-L F1'].tolist()

# BLEU por amostra (sacrebleu sentence-level)
bleu_metric = BLEU(effective_order=True)
bleu_per_sample = [
    bleu_metric.sentence_score(hyp, [ref]).score
    for hyp, ref in zip(generated_responses, reference_outputs)
]

df_results = pd.DataFrame({
    'Instrução'           : [s['Instruction'][:50] + '...' for s in samples],
    'PPL'                 : [round(p, 2) for p in ppl_scores],
    'BLEU'                : [round(b, 2) for b in bleu_per_sample],
    'ROUGE-1 F1'          : [round(r, 3) for r in rouge_f1_1],
    'ROUGE-2 F1'          : [round(r, 3) for r in rouge_f1_2],
    'ROUGE-L F1'          : [round(r, 3) for r in rouge_f1_L],
    'Faithfulness'        : [round(f, 3) for f in faithfulness_scores],
    'Answer Relevance'    : [round(r, 3) for r in relevance_scores],
    'Plan Adherence'      : [round(p, 3) for p in plan_scores],
})

# Linha de médias
means_row = pd.DataFrame([{
    'Instrução'        : '📊 MÉDIA',
    'PPL'              : round(np.mean(ppl_scores), 2),
    'BLEU'             : round(np.mean(bleu_per_sample), 2),
    'ROUGE-1 F1'       : round(np.mean(rouge_f1_1), 3),
    'ROUGE-2 F1'       : round(np.mean(rouge_f1_2), 3),
    'ROUGE-L F1'       : round(np.mean(rouge_f1_L), 3),
    'Faithfulness'     : round(np.mean(faithfulness_scores), 3),
    'Answer Relevance' : round(np.mean(relevance_scores), 3),
    'Plan Adherence'   : round(np.mean(plan_scores), 3),
}])

df_display = pd.concat([df_results, means_row], ignore_index=True)

def highlight_mean_row(row):
    if row['Instrução'] == '📊 MÉDIA':
        return ['background-color: #1a1a2e; color: #e0e0e0; font-weight: bold'] * len(row)
    return [''] * len(row)

styled = (
    df_display.style
    .apply(highlight_mean_row, axis=1)
    .background_gradient(subset=['ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1',
                                  'Faithfulness', 'Answer Relevance', 'Plan Adherence'],
                         cmap='YlGn', vmin=0, vmax=1)
    .background_gradient(subset=['PPL'], cmap='YlOrRd_r')
    .set_caption('📋 Resultados por Amostra — Todas as Métricas')
)

styled

## Visualizações

In [ ]:
# ============================================================
#  VISUALIZAÇÕES
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📊 Avaliação do Modelo Fine-Tuned — Visão Geral', fontsize=16, fontweight='bold', y=1.01)

sample_labels = [f'S{i+1}' for i in range(len(samples))]
colors = plt.cm.Set2.colors

# --- 1. Perplexidade por Amostra ---
ax = axes[0, 0]
bars = ax.bar(sample_labels, ppl_scores, color=colors[0], edgecolor='white', linewidth=0.5)
ax.axhline(np.mean(ppl_scores), color='red', linestyle='--', linewidth=1.5, label=f'Média: {np.mean(ppl_scores):.1f}')
ax.set_title('Perplexidade (PPL)', fontweight='bold')
ax.set_ylabel('PPL (menor = melhor)')
ax.set_xlabel('Amostra')
ax.legend(fontsize=9)

# --- 2. BLEU por Amostra ---
ax = axes[0, 1]
ax.bar(sample_labels, bleu_per_sample, color=colors[1], edgecolor='white', linewidth=0.5)
ax.axhline(np.mean(bleu_per_sample), color='red', linestyle='--', linewidth=1.5,
           label=f'Média: {np.mean(bleu_per_sample):.1f}')
ax.set_title('BLEU Score por Amostra', fontweight='bold')
ax.set_ylabel('BLEU (maior = melhor)')
ax.set_xlabel('Amostra')
ax.legend(fontsize=9)

# --- 3. ROUGE F1 Comparativo ---
ax = axes[0, 2]
x = np.arange(len(sample_labels))
w = 0.28
ax.bar(x - w, rouge_f1_1, w, label='ROUGE-1', color=colors[2], edgecolor='white')
ax.bar(x,     rouge_f1_2, w, label='ROUGE-2', color=colors[3], edgecolor='white')
ax.bar(x + w, rouge_f1_L, w, label='ROUGE-L', color=colors[4], edgecolor='white')
ax.set_title('ROUGE F1 por Amostra', fontweight='bold')
ax.set_ylabel('F1 Score')
ax.set_xlabel('Amostra')
ax.set_xticks(x)
ax.set_xticklabels(sample_labels)
ax.legend(fontsize=9)

# --- 4. Faithfulness, Relevance, Plan Adherence ---
ax = axes[1, 0]
ax.plot(sample_labels, faithfulness_scores, 'o-', color=colors[0], label='Faithfulness', linewidth=2)
ax.plot(sample_labels, relevance_scores, 's-', color=colors[1], label='Answer Relevance', linewidth=2)
ax.plot(sample_labels, plan_scores, '^-', color=colors[2], label='Plan Adherence', linewidth=2)
ax.set_ylim(0, 1.05)
ax.set_title('Métricas de Qualidade por Amostra', fontweight='bold')
ax.set_ylabel('Score (0–1)')
ax.set_xlabel('Amostra')
ax.legend(fontsize=9)

# --- 5. Radar / Spider Chart das Médias ---
ax = axes[1, 1]
ax.remove()
ax_radar = fig.add_subplot(2, 3, 5, projection='polar')

metric_names  = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Faithful', 'Relevance', 'Plan']
metric_values = [
    np.mean(rouge_f1_1),
    np.mean(rouge_f1_2),
    np.mean(rouge_f1_L),
    np.mean(faithfulness_scores),
    np.mean(relevance_scores),
    np.mean(plan_scores),
]

angles = np.linspace(0, 2 * np.pi, len(metric_names), endpoint=False).tolist()
values_closed = metric_values + [metric_values[0]]
angles_closed = angles + [angles[0]]

ax_radar.plot(angles_closed, values_closed, 'o-', linewidth=2, color=colors[3])
ax_radar.fill(angles_closed, values_closed, alpha=0.25, color=colors[3])
ax_radar.set_xticks(angles)
ax_radar.set_xticklabels(metric_names, size=9)
ax_radar.set_ylim(0, 1)
ax_radar.set_title('Radar das Médias (0–1)', fontweight='bold', pad=15)

# --- 6. Sumário em barras horizontais ---
ax = axes[1, 2]
summary_names = [
    'ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1',
    'Faithfulness', 'Answer Relevance', 'Plan Adherence'
]
summary_vals = metric_values
bar_colors   = [colors[i % len(colors)] for i in range(len(summary_names))]
h_bars = ax.barh(summary_names, summary_vals, color=bar_colors, edgecolor='white')
ax.set_xlim(0, 1)
ax.set_title('Resumo das Métricas (Médias)', fontweight='bold')
ax.set_xlabel('Score médio')
for bar, val in zip(h_bars, summary_vals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/avaliacao_Unicamp_metricas.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figura salva em avaliacao_metricas.png')

## Análise Qualitativa: Comparação Resposta vs. Referência

In [ ]:
# ============================================================
#  ANÁLISE QUALITATIVA
# ============================================================

INSPECT_INDICES = [0, 3, 6]   # Altere para ver outras amostras

for idx in INSPECT_INDICES:
    s   = samples[idx]
    gen = generated_responses[idx]

    print('=' * 70)
    print(f'🔎 AMOSTRA {idx + 1}')
    print(f'📌 Instrução : {s["Instruction"]}')
    #if s['Input']:
    #    print(f'📥 Contexto  : {s["Input"]}')
    print()
    print('✅ REFERÊNCIA:')
    print(s['Output'])
    print()
    print('🤖 GERADO:')
    print(gen if gen else '[sem saída gerada]')
    print()
    print(f'   PPL              = {ppl_scores[idx]:.2f}')
    print(f'   BLEU             = {bleu_per_sample[idx]:.2f}')
    print(f'   ROUGE-L F1       = {rouge_f1_L[idx]:.3f}')
    print(f'   Faithfulness     = {faithfulness_scores[idx]:.3f}')
    print(f'   Answer Relevance = {relevance_scores[idx]:.3f}')
    print(f'   Plan Adherence   = {plan_scores[idx]:.3f}')
    print()

## Relatório Final

In [ ]:
# ============================================================
#  RELATÓRIO FINAL AUTOMÁTICO
# ============================================================

def interpret(metric: str, value: float) -> str:
    thresholds = {
        'rouge'       : [(0.5, '🟢 Bom'), (0.3, '🟡 Moderado'), (0, '🔴 Fraco')],
        'faith_rel_pl': [(0.7, '🟢 Bom'), (0.4, '🟡 Moderado'), (0, '🔴 Fraco')],
        'bleu'        : [(30,  '🟢 Bom'), (15,  '🟡 Moderado'), (0, '🔴 Fraco')],
    }
    key = 'bleu' if metric == 'bleu' else (
          'rouge' if 'rouge' in metric else 'faith_rel_pl'
    )
    for threshold, label in thresholds[key]:
        if value >= threshold:
            return label
    return '🔴 Fraco'

mean_r1 = np.mean(rouge_f1_1)
mean_r2 = np.mean(rouge_f1_2)
mean_rL = np.mean(rouge_f1_L)
mean_bl = np.mean(bleu_per_sample)
mean_fa = np.mean(faithfulness_scores)
mean_re = np.mean(relevance_scores)
mean_pl = np.mean(plan_scores)

print('=' * 65)
print('         📋  RELATÓRIO DE AVALIAÇÃO — MODELO FINE-TUNED')
print('=' * 65)
print(f'  Modelo Base     : tucano')
print(f'  Adaptador       : LoRA  ({LORA_MODEL_PATH})')
print(f'  Dataset         : {DATASET_PATH}  ({len(samples)} amostras)')
print(f'  Dispositivo     : {DEVICE}')
print('=' * 65)
print()
print(f'  PERPLEXIDADE (PPL)')
print(f'    Média : {np.mean(ppl_scores):.2f}  —  PPL menor = modelo mais previsível')
print()
print(f'  BLEU')
print(f'    Corpus Score : {bleu_results["bleu_score"]:.2f}  {interpret("bleu", bleu_results["bleu_score"])}')
print(f'    Média por amostra : {mean_bl:.2f}')
print()
print(f'  ROUGE')
print(f'    ROUGE-1 F1 : {mean_r1:.3f}  {interpret("rouge1", mean_r1)}')
print(f'    ROUGE-2 F1 : {mean_r2:.3f}  {interpret("rouge2", mean_r2)}')
print(f'    ROUGE-L F1 : {mean_rL:.3f}  {interpret("rougeL", mean_rL)}')
print()
print(f'  FIDELIDADE (FAITHFULNESS)')
print(f'    Média : {mean_fa:.3f}  {interpret("faithfulness", mean_fa)}')
print()
print(f'  RELEVÂNCIA DA RESPOSTA (ANSWER RELEVANCE)')
print(f'    Média : {mean_re:.3f}  {interpret("relevance", mean_re)}')
print()
print(f'  ADERÊNCIA AO PLANO (PLAN ADHERENCE)')
print(f'    Média : {mean_pl:.3f}  {interpret("plan", mean_pl)}')
print()
print('─' * 65)
print('  💡 ANÁLISE GLOBAL:')
print(f'     O modelo apresenta PPL={np.mean(ppl_scores):.1f}, indicando',
      'boa' if np.mean(ppl_scores) < 50 else 'moderada', 'fluência.')
print(f'     As métricas de sobreposição léxica (BLEU/ROUGE) refletem')
print(f'     o grau de fidelidade ao texto de referência.')
print(f'     Faithfulness={mean_fa:.2f} e Plan Adherence={mean_pl:.2f} indicam')
print(f'     se o modelo é adequado para pipelines RAG e agentes.')
print('=' * 65)

## Exportar Resultados

In [ ]:
# ============================================================
#  EXPORTAR RESULTADOS
# ============================================================

# CSV detalhado com respostas geradas
df_export = df_results.copy()
df_export['Gerado']    = generated_responses
df_export['Referência'] = reference_outputs

CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/resultados_Unicamp_avaliacao.csv'
df_export.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')

print(f'✅ Resultados exportados:')
print(f'   📄 {CSV_PATH}           — métricas por amostra + textos')
print(f'   🖼️  avaliacao_metricas.png — visualizações')
print()
print('🎉 Avaliação concluída! Todos os artefatos foram gerados.')